# 📘 Agentic Architectures 11 (Agno): Meta-Controller

This notebook is the **Agno-framework** counterpart of `11_meta_controller.ipynb`. The original notebook wires a supervisory router and three specialists (Generalist / Researcher / Coder) as a LangGraph `StateGraph` with a fan-out conditional edge from the controller to one of three specialist nodes.

In Agno we mirror the **exact pattern used in production** at `agent_service/app/agents/router.py`:

- The **router** is one `Agent` with `output_schema=ControllerDecision` + `use_json_mode=True` — a single LLM call that emits a structured routing decision
- The three **specialists** are three plain `Agent` instances (the Researcher gets `TavilyTools`)
- A tiny Python dispatch function reads the structured decision and forwards the query to the chosen specialist

The whole orchestration is ~5 lines of Python instead of a `StateGraph` with `add_conditional_edges`.

### Mapping the abstraction

| Concern | LangGraph (`11_meta_controller.ipynb`) | Agno (this notebook) |
|---|---|---|
| Controller | `meta_controller_node` calling `llm.with_structured_output(ControllerDecision)` | One `Agent` with `output_schema=ControllerDecision` + `use_json_mode=True` |
| Specialists | Three nodes built via `create_specialist_node(...)` | Three separate `Agent` instances |
| Routing logic | `add_conditional_edges(..., route_to_specialist, {...})` | A Python `dict` lookup |
| Graph state | `MetaAgentState` `TypedDict` | Plain function arguments (`query`, `decision`) |
| Driver | `meta_agent.invoke({...})` | A small `run_agent(query)` function |

The user-facing behaviour is identical: greeting → Generalist, current-events question → Researcher (which uses the search tool), code request → Coder.

## Phase 0: Foundation & Setup

In [ ]:
# !pip install -q -U agno openai python-dotenv rich pydantic

In [ ]:
import os
from typing import Any
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from agno.agent import Agent
from agno.models.openai import OpenAILike
from agno.tools.tavily import TavilyTools

from rich.console import Console
from rich.markdown import Markdown

load_dotenv()

for key in ["SILICONFLOW_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

console = Console()
print("Environment variables loaded.")

## Phase 1: Shared LLM Configuration

Same SiliconFlow endpoint used by `03_ReAct_agno.ipynb`. We share one model instance across the controller and the three specialists — production code does the same (`agent_service`'s `ProviderManager` hands out a singleton).

In [ ]:
model = OpenAILike(
    id="deepseek-ai/DeepSeek-V3",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
    temperature=0,
)

print(f"Shared model: {model.id}")

## Phase 2: Building the Specialist Agents

Three plain Agno agents, each with its own persona. Only the Researcher gets a tool — exactly mirroring the original notebook.

In [ ]:
generalist = Agent(
    name="Generalist",
    model=model,
    instructions=[
        "You are a friendly and helpful generalist AI assistant.",
        "You handle casual conversation and simple questions.",
        "Respond directly and concisely.",
    ],
    markdown=True,
    telemetry=False,
)

researcher = Agent(
    name="Researcher",
    model=model,
    tools=[
        TavilyTools(
            api_key=os.environ.get("TAVILY_API_KEY"),
            search_depth="advanced",
            format="markdown",
            include_answer=True,
        )
    ],
    instructions=[
        "You are an expert researcher.",
        "You must use your search tool to find information to answer the user's question.",
        "After searching, synthesise a concise factual answer grounded in the search results.",
    ],
    add_datetime_to_context=True,
    markdown=True,
    telemetry=False,
    tool_call_limit=4,
)

coder = Agent(
    name="Coder",
    model=model,
    instructions=[
        "You are an expert Python programmer.",
        "Write clean, efficient Python code based on the user's request.",
        "Provide only the code, wrapped in markdown code blocks, with minimal explanation.",
    ],
    markdown=True,
    telemetry=False,
)

print("Specialists built: Generalist, Researcher, Coder")

## Phase 3: Building the Meta-Controller

This is the brain of the system. Following `agent_service/app/agents/router.py` exactly:

1. Define a Pydantic schema describing the decision
2. Build an `Agent` with `output_schema=...` + `use_json_mode=True` — Agno will force the LLM to emit a single structured JSON object conforming to the schema
3. Read `response.content` after `run()` — it will be a populated `ControllerDecision` instance, not a string

The structured-output discipline is the production-grade equivalent of LangGraph's `llm.with_structured_output(ControllerDecision)`.

In [ ]:
class ControllerDecision(BaseModel):
    """Schema for the meta-controller's routing decision."""
    next_agent: str = Field(
        description="The name of the specialist agent to call next. Must be one of ['Generalist', 'Researcher', 'Coder']."
    )
    reasoning: str = Field(description="A brief reason for choosing the next agent.")

specialist_catalog = {
    "Generalist": "Handles casual conversation, greetings, and simple questions.",
    "Researcher": "Answers questions about recent events, complex topics, or anything requiring up-to-date information from the web.",
    "Coder": "Writes Python code based on a user's specification.",
}
specialist_descriptions = "\n".join(f"- {name}: {desc}" for name, desc in specialist_catalog.items())

controller_instruction = (
    "You are the meta-controller for a multi-agent AI system. "
    "Your job is to analyse the user's request and route it to the most appropriate specialist agent.\n\n"
    "Available specialists:\n"
    f"{specialist_descriptions}\n\n"
    "Choose the best specialist and respond with the required structured decision."
)

meta_controller = Agent(
    name="Meta-Controller",
    model=model,
    instructions=[controller_instruction],
    output_schema=ControllerDecision,
    use_json_mode=True,
    add_datetime_to_context=True,
    telemetry=False,
)

print("Meta-controller built with structured output.")

**Discussion of the difference:**

- **LangGraph:** the controller is one *node* in a graph; the *graph topology* (entry point + conditional edges) wires it to specialists.
- **Agno:** the controller is one *agent*; the *Python dispatch code* wires it to specialists.

Both approaches achieve the same routing semantics. The trade-off lines up with the trade-offs in `02` and `03`: LangGraph makes the wiring visible and editable as data, Agno makes the wiring a one-liner of code. For a 3-way fan-out like this the Agno expression is shorter; for a 20-node graph with shared sub-paths and back-edges, LangGraph's explicit topology becomes the cleaner abstraction.

## Phase 4: Assembling and Running the System

This is the entire orchestration layer — what LangGraph builds with `StateGraph(...).add_conditional_edges(...)` is here a single function with a `dict` lookup.

In [ ]:
specialists = {
    "Generalist": generalist,
    "Researcher": researcher,
    "Coder": coder,
}

def run_meta_agent(query: str) -> None:
    console.print("--- 🧠 Meta-Controller Analysing Request ---")
    decision_response = meta_controller.run(query)
    decision: ControllerDecision = decision_response.content
    console.print(
        f"[yellow]Routing decision:[/yellow] Send to [bold]{decision.next_agent}[/bold]. "
        f"[italic]Reason: {decision.reasoning}[/italic]"
    )

    specialist = specialists.get(decision.next_agent)
    if specialist is None:
        console.print(f"[red]Unknown specialist '{decision.next_agent}' — falling back to Generalist[/red]")
        specialist = specialists["Generalist"]

    result = specialist.run(query)
    console.print("\n[bold]Final Response:[/bold]")
    console.print(Markdown(result.content or "(no content)"))

print("Meta-controller dispatch ready.")

## Phase 5: Demonstration

Three test queries, one for each specialist — same as the original notebook.

In [ ]:
console.print("--- 💬 Test 1: General Conversation ---")
run_meta_agent("Hello, how are you today?")

console.print("\n--- 🔬 Test 2: Research Question ---")
run_meta_agent("What were NVIDIA's latest financial results?")

console.print("\n--- 💻 Test 3: Coding Request ---")
run_meta_agent("Can you write me a python function to calculate the nth fibonacci number?")

## Conclusion

We re-implemented the **Meta-Controller** architecture in Agno using the same pattern the production `agent_service/app/agents/router.py` uses: one `Agent` with structured output decides which specialist to invoke, and a tiny Python dispatcher does the actual fan-out. The three specialists are plain `Agent` instances — no graph wiring needed.

Structurally, the Meta-Controller is where Agno's high-level abstraction starts to feel *natural* rather than just *concise*: a router + specialists is the canonical shape that Agno was designed around, and the production code base demonstrates this scales to many specialists, shared model pools, streaming events, and rate-limit retries without changing the basic shape.

For benchmark purposes, this notebook plus the LangGraph original form a clean A/B: same controller logic, same specialists, same queries — only the orchestrator changes.